# VisTacFusion Fine-Tune (Colab)

Fine-tune a sim-pretrained model on real data.

**Pipeline:** Sim-only pretrain (done) → Load checkpoint → Fine-tune on real data (this notebook)

The sim-pretrained model already has:
- DPT decoder trained on sim depth/normal
- Fusion trunk + pose head trained on sim pose (both/rgb mode ~4°)
- Frozen T3 (tactile) + MAE (RGB) encoders

Fine-tuning adapts the trainable components to real data with a lower learning rate.

## 1. Environment Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/cynthiahuang1004/VisTacFusion.git
%cd VisTacFusion
!git checkout VisTacFusion-v2

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers safetensors tensorboard opencv-python matplotlib pyyaml tqdm timm

import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)')

## 2. Download / Cache Pretrained Encoders

In [ ]:
import os, shutil

DRIVE_ENCODER_DIR = '/content/drive/MyDrive/VisTacFusion_encoders'
LOCAL_ENCODER_DIR = 'pretrained_encoders'

os.makedirs(DRIVE_ENCODER_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_ENCODER_DIR}/t3_large', exist_ok=True)
os.makedirs(f'{LOCAL_ENCODER_DIR}/t3_large', exist_ok=True)

ENCODER_FILES = {
    't3_large/encoder_mini.pth': 'https://huggingface.co/datasets/alanz-mit/FoundationTactile/resolve/main/models/t3_large/encoders/mini.pth',
    't3_large/trunk.pth': 'https://huggingface.co/datasets/alanz-mit/FoundationTactile/resolve/main/models/t3_large/trunk.pth',
    'mae_vitl16.pth': 'https://dl.fbaipublicfiles.com/mae/pretrain/mae_pretrain_vit_large.pth',
}

for local_name, url in ENCODER_FILES.items():
    drive_path = f'{DRIVE_ENCODER_DIR}/{local_name}'
    local_path = f'{LOCAL_ENCODER_DIR}/{local_name}'
    
    if os.path.exists(drive_path):
        size_mb = os.path.getsize(drive_path) / 1e6
        print(f'[cached] {local_name} ({size_mb:.0f} MB) <- Drive')
    else:
        print(f'[downloading] {local_name} ...')
        !wget -q --show-progress -O "{drive_path}" "{url}"
        size_mb = os.path.getsize(drive_path) / 1e6
        print(f'  saved to Drive ({size_mb:.0f} MB)')
    
    if os.path.exists(local_path):
        os.remove(local_path)
    os.symlink(drive_path, local_path)

print('\nEncoder files ready.')

## 3. Load Sim-Pretrained Checkpoint

Upload your sim-pretrained checkpoint, or point to it on Google Drive.

Available checkpoints from sim-only training:
- `best_pose.pt` — best pose validation (recommended for finetune)
- `best_depth.pt` — best depth validation
- `latest.pt` — last epoch

In [ ]:
# ===================== SET CHECKPOINT PATH =====================

# Option A: From Google Drive
SIM_CHECKPOINT = ''   # e.g. '/content/drive/MyDrive/VisTacFusion_checkpoints/sim190_simonly/best_pose.pt'

# Option B: Upload directly
# from google.colab import files
# uploaded = files.upload()  # upload best_pose.pt
# SIM_CHECKPOINT = list(uploaded.keys())[0]

# ==============================================================

if SIM_CHECKPOINT and os.path.exists(SIM_CHECKPOINT):
    size_mb = os.path.getsize(SIM_CHECKPOINT) / 1e6
    print(f'Checkpoint: {SIM_CHECKPOINT} ({size_mb:.0f} MB)')
else:
    print('WARNING: Checkpoint not found. Set SIM_CHECKPOINT path above.')

## 4. Set Real Data Path

In [ ]:
# ===================== SET DATA PATHS =====================

REAL_ROOT = ''          # e.g. '/content/drive/MyDrive/real_data'
MESH_DIR = ''           # e.g. '/content/drive/MyDrive/meshes'

# Optional: include sim data for co-training finetune
SIM_ROOT = ''           # leave empty for real-only finetune

# ==============================================================

for name, path in [('REAL_ROOT', REAL_ROOT), ('SIM_ROOT', SIM_ROOT), ('MESH_DIR', MESH_DIR)]:
    if path:
        exists = os.path.isdir(path)
        n = len(os.listdir(path)) if exists else 0
        print(f'{name}: {path} ({"OK" if exists else "NOT FOUND"}, {n} items)')
    else:
        print(f'{name}: (not set)')

## 5. Configure Fine-Tuning

In [ ]:
import yaml

# ===================== FINETUNE SETTINGS =====================

FINETUNE_LR = 2e-5              # 10x lower than pretraining
FINETUNE_EPOCHS = 150
BATCH_SIZE = 32
WARMUP_STEPS = 500
REAL_VAL_EVERY = 10

# Model config
MODEL_CONFIG = 'ablation/encoder/tac_t3_rgb_mae.yaml'

# Output
OUTPUT_DIR = 'outputs/colab_finetune'

# ==============================================================

# Generate data config
if SIM_ROOT:
    mode = 'sim+real'
else:
    mode = 'sim+real'  # still sim+real mode, but sim samples = 0

data_cfg = {
    'image_size': 224,
    'dataset': mode,
    'synthetic': {'num_samples': 256, 'num_objects': 8},
    'sim': {
        'root': SIM_ROOT if SIM_ROOT else REAL_ROOT,  # placeholder if no sim
        'mesh_dir': MESH_DIR,
        'rgb_subdir': 'rgb',
        'use_gt_depth': True,
        'use_rendered_normals': True if SIM_ROOT else False,
        'gel_view_m': 0.017502,
        'rot_augment': True,
        'rot_augment_max_deg': 180.0,
        'val_every': 20,
        'train_samples_per_session': 0 if not SIM_ROOT else None,
    },
    'real': {
        'root': REAL_ROOT,
        'mesh_dir': MESH_DIR,
        'rgb_subdir': 'rgb',
        'use_rendered_normals': False,
        'val_every': REAL_VAL_EVERY,
        'augment': False,
        'oversample': 1,
    },
    'loader': {
        'num_workers': 4,
        'pin_memory': True,
        'prefetch_factor': 4,
        'persistent_workers': True,
    },
    'norm': {
        'imagenet_mean': [123.675, 116.28, 103.53],
        'imagenet_std': [58.395, 57.12, 57.375],
    },
}

# Remove None values
if data_cfg['sim']['train_samples_per_session'] is None:
    del data_cfg['sim']['train_samples_per_session']

data_config_path = 'configs/data_finetune_colab.yaml'
with open(data_config_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

# Generate train config
train_cfg = {
    'seed': 0,
    'device': 'cuda',
    'amp': True,
    'optim': {
        'name': 'adamw',
        'lr': FINETUNE_LR,
        'weight_decay': 0.05,
        'betas': [0.9, 0.999],
    },
    'schedule': {
        'name': 'cosine',
        'warmup_steps': WARMUP_STEPS,
        'max_epochs': FINETUNE_EPOCHS,
    },
    'batch_size': BATCH_SIZE,
    'num_workers': 4,
    'modality_dropout': {
        'enabled': True,
        'p_both': 0.55,
        'p_tactile_only': 0.35,
        'p_rgb_only': 0.10,
        'p_dpt_inject': 0.5,
    },
    'loss': {
        'depth': {'type': 'mse', 'grad_matching_weight': 0.0, 'weight': 1.0},
        'normal': {'type': 'mse', 'weight': 1.0},
        'pose': {'rot_weight': 1.0, 'trans_weight': 1.0, 'weight': 1.0},
        'uncertainty_weighting': True,
        'grouped_uncertainty': True,
        'dense_pose_ratio': 1.0,
    },
    'augmentation': {
        'tactile': {'gain': False, 'bias': False, 'gradient': False, 'residual_noise': False, 'geometric': False},
        'rgb': {'photometric': False, 'geometric': False},
    },
    'eval': {
        'metrics': ['depth_absrel', 'depth_rmse', 'normal_mean_angle', 'pose_rot_deg', 'pose_trans_mm'],
        'report_per_config': True,
    },
    'log_every': 50,
    'ckpt_every_epochs': 5,
}

train_config_path = 'configs/train_finetune_colab.yaml'
with open(train_config_path, 'w') as f:
    yaml.dump(train_cfg, f, default_flow_style=False, sort_keys=False)

print(f'Model:      {MODEL_CONFIG}')
print(f'Checkpoint: {SIM_CHECKPOINT}')
print(f'Data:       {data_config_path}')
print(f'Train:      {train_config_path}')
print(f'Output:     {OUTPUT_DIR}')
print(f'LR:         {FINETUNE_LR}')
print(f'Epochs:     {FINETUNE_EPOCHS}')

## 6. Verify Setup

In [ ]:
import sys
sys.path.insert(0, '.')

from vistacfusion.engine.train import merge_configs
from vistacfusion.models.model import build_model
from vistacfusion.data.dataset import build_datasets

cfg = merge_configs(MODEL_CONFIG, train_config_path, data_config_path)

train_ds, val_ds = build_datasets(cfg)
print(f'Train: {len(train_ds)} samples')
print(f'Val:   {len(val_ds)} samples')

model = build_model(cfg)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {type(model).__name__} ({total/1e6:.1f}M total, {trainable/1e6:.1f}M trainable)')

# Test checkpoint loading
if SIM_CHECKPOINT:
    from vistacfusion.engine.train import load_checkpoint
    load_checkpoint(SIM_CHECKPOINT, model, device=torch.device('cpu'))
    print(f'Checkpoint loaded OK (finetune mode: fresh optimizer)')

del model, train_ds, val_ds
torch.cuda.empty_cache()
print('Setup verified!')

## 7. Fine-Tune

In [ ]:
!python -u -m vistacfusion.engine.train \
  --model {MODEL_CONFIG} \
  --train {train_config_path} \
  --data {data_config_path} \
  --output-dir {OUTPUT_DIR} \
  --finetune \
  --resume {SIM_CHECKPOINT}

## 8. Evaluate Best Checkpoints

Reports: depth MSE, normal MSE, rot L1, rot (deg), trans L1 for each modality config (both / tactile / rgb).

In [ ]:
import sys, math
sys.path.insert(0, '.')
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from vistacfusion.engine.train import merge_configs, load_checkpoint
from vistacfusion.models.model import build_model
from vistacfusion.data.dataset import build_datasets
from vistacfusion.engine.eval import precompute_encoder_cache, _slice_cache

def evaluate_model(output_dir, model_config, train_config, data_config, device='cuda:0'):
    cfg = merge_configs(model_config, train_config, data_config)
    dev = torch.device(device)
    _, val_ds = build_datasets(cfg)
    val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

    mt = cfg.get('model_type', 'visuo_tactile')
    configs = ('tactile',) if mt in ('single_encoder', 'mvitac') else ('both', 'tactile', 'rgb')

    ckpts = {}
    for name in ['best_depth', 'best_pose', 'latest']:
        path = os.path.join(output_dir, f'{name}.pt')
        if os.path.exists(path):
            ckpts[name] = path
    if not ckpts:
        print(f'No checkpoints found in {output_dir}')
        return

    for ckpt_name, ckpt_path in ckpts.items():
        print(f'\n{"="*60}')
        print(f'Checkpoint: {ckpt_name}')
        print(f'{"="*60}')

        model = build_model(cfg).to(dev)
        load_checkpoint(ckpt_path, model, device=dev)
        model.eval()

        enc_cache = None
        if hasattr(model, 'tactile_encoder'):
            enc_cache = precompute_encoder_cache(model, val_loader, dev)

        acc = {c: {'depth_mse': 0., 'normal_mse': 0., 'rot_l1': 0.,
                    'rot_deg': 0., 'trans_l1': 0., 'n': 0} for c in configs}

        sample_idx = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in batch.items()}
                bs = batch['rgb'].shape[0]
                batch_enc = _slice_cache(enc_cache, sample_idx, sample_idx + bs, dev) if enc_cache else None
                sample_idx += bs

                gt_pose = batch['pose']

                for c in configs:
                    domain_ids = batch.get('domain')
                    if domain_ids is None:
                        domain_ids = torch.ones(bs, dtype=torch.long, device=dev)
                    out = model(batch['rgb'], batch['tactile'], config=c,
                                encoder_cache=batch_enc,
                                object_ids=batch.get('object'),
                                domain_ids=domain_ids)
                    a = acc[c]
                    a['depth_mse'] += F.mse_loss(out['depth'], batch['depth']).item() * bs
                    a['normal_mse'] += F.mse_loss(out['normal'], batch['normal']).item() * bs

                    se2 = out['se2']
                    a['rot_l1'] += (se2[:,:2] - gt_pose[:,:2]).abs().sum(dim=-1).mean().item() * bs
                    theta_pred = torch.atan2(se2[:,1], se2[:,0])
                    theta_gt = torch.atan2(gt_pose[:,1], gt_pose[:,0])
                    rot_err = torch.abs(theta_pred - theta_gt)
                    rot_err = torch.min(rot_err, 2*math.pi - rot_err)
                    a['rot_deg'] += (rot_err * 180 / math.pi).mean().item() * bs
                    a['trans_l1'] += (se2[:,2:] - gt_pose[:,2:]).abs().mean().item() * bs
                    a['n'] += bs

        header = f'{"Config":>10s}  {"Depth MSE":>10s}  {"Normal MSE":>10s}  {"Rot L1":>8s}  {"Rot (deg)":>10s}  {"Trans L1":>8s}'
        print(header)
        print('-' * len(header))
        for c in configs:
            a = acc[c]
            n = a['n']
            print(f'{c:>10s}  {a["depth_mse"]/n:10.6f}  {a["normal_mse"]/n:10.6f}  '
                  f'{a["rot_l1"]/n:8.4f}  {a["rot_deg"]/n:10.3f}°  {a["trans_l1"]/n:8.4f}')

        del model
        torch.cuda.empty_cache()

evaluate_model(OUTPUT_DIR, MODEL_CONFIG, train_config_path, data_config_path)

## 9. Plot Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

history_path = f'{OUTPUT_DIR}/history.json'
if not os.path.exists(history_path):
    print(f'{history_path} not found.')
else:
    with open(history_path) as f:
        history = json.load(f)
    
    epochs = [h['epoch'] for h in history]
    cfg_key = 'both' if 'both' in history[0]['val'] else 'tactile'
    
    metrics = [
        ('depth_mse', 'Depth MSE', '#2a78d6'),
        ('normal_mse', 'Normal MSE', '#1baf7a'),
        ('pose_rot_deg', 'Rotation (deg)', '#eb6834'),
        ('pose_rot_l1', 'Rotation L1', '#e87ba4'),
        ('pose_trans', 'Translation L1', '#4a3aa7'),
    ]
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'Fine-tune: sim-pretrained → real (epoch {epochs[-1]})', fontsize=14, fontweight='bold')
    
    for ax, (key, label, color) in zip(axes.flat, metrics):
        vals = [h['val'][cfg_key].get(key) for h in history]
        vals = [v for v in vals if v is not None]
        if vals:
            ax.plot(epochs[:len(vals)], vals, color=color, lw=1.8)
            best_i = int(np.argmin(vals))
            ax.scatter(epochs[best_i], vals[best_i], color=color, s=60, zorder=5)
            ax.set_title(f'{label} (best: {vals[best_i]:.4f} @ e{epochs[best_i]})', fontsize=11)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
    
    axes.flat[-1].axis('off')
    plt.tight_layout()
    plt.show()
    
    last = history[-1]['val'][cfg_key]
    print(f"\nFinal (epoch {epochs[-1]}):")
    for key, label, _ in metrics:
        if key in last:
            print(f'  {label:20s} {last[key]:.6f}')

In [ ]:
DRIVE_SAVE_DIR = f'/content/drive/MyDrive/VisTacFusion_checkpoints/{os.path.basename(OUTPUT_DIR)}'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

for name in ['best_depth.pt', 'best_pose.pt', 'latest.pt', 'history.json']:
    src = os.path.join(OUTPUT_DIR, name)
    if os.path.exists(src):
        dst = os.path.join(DRIVE_SAVE_DIR, name)
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'  {name} ({size_mb:.0f} MB)')
    else:
        print(f'  {name} (not found)')

print(f'\nSaved to {DRIVE_SAVE_DIR}/')

## 10. Save Checkpoints to Drive